Approach 2 where I extract K and n infinity from the 6:2 experiment and then use it to find ntot to the find pressure for new gepomretry of Vsol to Vg.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ======= USER INPUTS =======
kinetic_csv = r"C:\Users\chand\Documents\GitHub\Thesis\CSV files outputs\Modified vant hoff modelling\ntot values.csv"
time_col_candidates = ["Elapsed Time (h)"]
ntot_col_candidates = ["n_total (µmol)"]
# ===========================

# --- load ---
df = pd.read_csv(kinetic_csv)
df.columns = [c.strip() for c in df.columns]

def pick_col(cands, cols):
    for c in cands:
        if c in cols: return c
    raise KeyError(f"Column not found. Tried: {cands}. Got: {list(cols)}")

t_col = pick_col(time_col_candidates, df.columns)
n_col = pick_col(ntot_col_candidates,  df.columns)

t_h    = pd.to_numeric(df[t_col], errors="coerce")
n_umol = pd.to_numeric(df[n_col], errors="coerce")
mask   = t_h.notna() & n_umol.notna()
t_h, n_umol = t_h[mask].reset_index(drop=True), n_umol[mask].reset_index(drop=True)
t_h = t_h - t_h.iloc[0]  # start at 0 h

# --- model: n(t) = n0 + n_inf * (1 - exp(-k t)) ---
def n_model(t, n_inf, k, n0):
    return n0 + n_inf * (1 - np.exp(-k*t))

# --- fit (SciPy if available; otherwise light grid) ---
try:
    from scipy.optimize import curve_fit
    p0 = [max(n_umol.iloc[-1], 1e-6), 0.01, max(n_umol.iloc[0], 0.0)]   # n_inf[µmol], k[1/h], n0[µmol]
    bounds = ([0.0, 0.0, 0.0], [np.inf, np.inf, np.inf])
    popt, pcov = curve_fit(n_model, t_h.values, n_umol.values, p0=p0, bounds=bounds, maxfev=20000)
    n_inf_umol, k_fit, n0_umol = popt
except Exception as e:
    print("curve_fit unavailable; doing a small grid search...", e)
    n_inf_umol = float(n_umol.iloc[-1])
    n0_umol    = float(n_umol.iloc[0])
    k_grid = np.logspace(-4, 0, 120)
    best = (np.inf, None)
    for k_try in k_grid:
        pred = n_model(t_h.values, n_inf_umol, k_try, n0_umol)
        sse  = np.sum((n_umol.values - pred)**2)
        if sse < best[0]:
            best = (sse, k_try)
    k_fit = best[1]

# fitted curve
n_fit_umol = n_model(t_h.values, n_inf_umol, k_fit, n0_umol)

print(f"Fitted k = {k_fit:.5f} 1/h")
print(f"Fitted n_inf = {n_inf_umol:.3f} µmol   (asymptotic total O₂)")
print(f"Fitted n0 = {n0_umol:.3f} µmol")

# (optional) add to df
df_fit = pd.DataFrame({"Elapsed Time (h)": t_h, "n_total (µmol)": n_umol, "n_fit (µmol)": n_fit_umol})

# quick plot
plt.figure(figsize=(8.2,4.6))
plt.plot(t_h, n_umol,  lw=2, label="data")
plt.plot(t_h, n_fit_umol, "--", lw=2, label=f"fit (k={k_fit:.4f} 1/h)")
plt.xlabel("Elapsed Time (h)"); plt.ylabel("n_total (µmol)")
plt.title("First-order fit to n_total(t)")
plt.grid(True, alpha=0.3); plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ======= USER GEOMETRY / PHYSICS =======
Vsol_mL = 7.0     # change as needed
Vg_mL   = 2.0     # change as needed
T_K     = 297.15  # K
R       = 0.082057366079      # L·atm·mol^-1·K^-1
kH      = 1.3e-3              # mol·L^-1·atm^-1
k_fit = 0.00873 
n_inf_umol = 6.257
# =======================================

def predict_pressure_kPa(t_hours, n_inf_umol, k_fit, n0_umol, Vsol_mL, Vg_mL, T_K, R, kH):
    # n(t) from fitted params (convert µmol -> mol)
    n_t_mol = (n0_umol + n_inf_umol*(1 - np.exp(-k_fit*t_hours))) * 1e-6
    # geometry factor A (mol/atm)
    Vsol_L = Vsol_mL/1000.0; Vg_L = Vg_mL/1000.0
    A = Vg_L/(R*T_K) + kH*Vsol_L
    P_atm = n_t_mol / A
    return P_atm * 101.325

# compute and plot
P_kPa = predict_pressure_kPa(t_h.values, n_inf_umol, k_fit, n0_umol, Vsol_mL, Vg_mL, T_K, R, kH)

plt.figure(figsize=(8.2,4.6))
plt.plot(t_h, P_kPa, lw=2)
plt.xlabel("Elapsed Time (h)"); plt.ylabel("Predicted Pressure (kPa)")
plt.title(f"Predicted headspace pressure | Vsol={Vsol_mL:.1f} mL, Vg={Vg_mL:.1f} mL")
plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

print(max(P_kPa))
print(P_kPa[-1]) #the very last value
print(t_h.values[-1]) #the time at the very last value
